# Análisis exploratorio (EDA)

Con este notebook se trata de describir los datos, sin ninguna transformación permanente.

**Salidas**

| Artefacto | Dónde |
|---|---|
| Figuras | `reports/figures/eda/<capa>/` |
| Tabla resumen por variable | `reports/eda/<capa>/variable_summary.csv` |

La capa de entrada es un parámetro (`LAYER`). La idea es pasar el notebook dos
veces: primero sobre la capa aún sin limpiar, que es lo que dice qué preprocesado
hace falta, y después sobre `silver`, para ver qué acaba entrando al modelo. Sin
la primera pasada el preprocesado se diseña a ciegas; sin la segunda no hay forma
de saber si arregló lo que pretendía. Las salidas van separadas por capa
justamente para poder ponerlas una al lado de la otra.

La segunda pasada es una comprobación, no un sitio donde tomar decisiones nuevas:
cualquier parámetro que se ajuste a partir de los datos (umbrales, imputación,
escalado) sale del split de entrenamiento, no de estas figuras.

**Qué se irá a `src/` cuando esté estable:** las funciones marcadas con
`# -> src/packagename/` más abajo. Mientras el criterio siga cambiando se quedan
aquí, porque mover código que aún no ha decidido qué hace sólo reparte la duda
entre dos ficheros.

## Definición de parámetros

Todo lo que habrá que cambiar para analizar otro dataset u otra variable objetivo se agrupa en esta celda.

In [ ]:
# Capa y tabla de entrada, dentro de `data/`. El notebook está pensado para
# ejecutarse dos veces: sobre la capa todavía sin limpiar, para decidir qué
# preprocesado hace falta, y sobre `silver`, para comprobar qué acaba entrando al
# modelo. Cambiar `LAYER` es lo único que separa una pasada de la otra, y cada
# una escribe en su propia carpeta para poder compararlas.
#
# Sea cual sea la capa, la tabla tiene que ser tabular: una fila por instante (y
# por punto, si hay varios), una columna por variable. Un campo gridded en
# NetCDF no vale aquí -- ese es el inventario previo, no este EDA.
LAYER = "silver"
DATASET = "measurements.parquet"

# Variable objetivo. 
TARGET = "swh"

# Columna temporal, y columna que separa puntos/estaciones si el dataset tiene
# más de uno. `GROUP = None` significa una sola serie.
TIME = "time"
GROUP = None

# Campo gridded opcional, relativo a `data/raw`, para la sección espacial.
# `None` la salta por completo.
GRID_FILE = None

# Rangos físicamente admisibles, en las unidades del dataset. Lo que caiga fuera
# no es un outlier: es un error. Rellenar sólo las variables de las que se sabe
# el rango -- un rango inventado es peor que ninguno.
PHYSICAL_RANGES: dict[str, tuple[float, float]] = {
    "swh": (0.0, 30.0),  # m
    "mwp": (0.0, 30.0),  # s
    "mwd": (0.0, 360.0),  # grados
    "msl": (85000.0, 110000.0),  # Pa
    "sst": (270.0, 320.0),  # K
    "u10": (-60.0, 60.0),  # m/s
    "v10": (-60.0, 60.0),  # m/s
}

# Umbral para "casi constante": fracción del valor más frecuente por encima de
# la cual la variable no aporta variabilidad utilizable.
NEAR_CONSTANT = 0.99

# Lags (en pasos del índice) explorados en la correlación cruzada. Las variables
# atmosféricas actúan sobre el oleaje con retardo, así que el lag 0 rara vez es
# el informativo.
MAX_LAG = 24

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import signal, stats
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from sklearn.feature_selection import mutual_info_regression
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf, pacf

from packagename import get_settings, set_seed, setup_logging
from packagename.etl import read_table, write_table
from packagename.viz import COLOR_NAMES, NEUTRALS, apply_style, remove_grid, savefig

setup_logging(level="INFO")
apply_style("paper")

settings = get_settings()
set_seed(settings.random_seed)
settings.paths.ensure()

# Subcarpetas propias, para que las figuras del EDA no se mezclen con las de los
# modelos. `savefig` resuelve rutas relativas bajo `paths.figures`, así que basta
# con el prefijo. La capa forma parte de la ruta: si las dos pasadas escribieran
# en el mismo sitio, la segunda borraría la evidencia que motivó el preprocesado.
FIG = f"eda/{LAYER}"
TABLES = settings.paths.reports / "eda" / LAYER
TABLES.mkdir(parents=True, exist_ok=True)

## Carga

`read_table` elige el lector por la extensión y las rutas de `settings` son
absolutas, así que esto funciona igual desde `notebooks/`, desde un script o
desde un job de Hydra.

In [ ]:
table = read_table(settings.paths.layer(LAYER) / DATASET)
table[TIME] = pd.to_datetime(table[TIME])

# El índice temporal es lo que hace que todo lo demás (resampleos, ACF, STL)
# signifique algo, así que se establece una vez y no se vuelve a tocar.
data = table.set_index(TIME).sort_index()

if GROUP is not None:
    print(f"{data[GROUP].nunique()} puntos; las secciones univariantes usan el primero")

numeric = data.select_dtypes(include="number")
features = [column for column in numeric.columns if column != TARGET]

print(f"{LAYER}/{DATASET}")
print(f"{len(data):,} filas x {len(numeric.columns)} variables numéricas")
print(f"{data.index.min()} -> {data.index.max()}")
data.head()

## 1. Calidad de los datos

El orden aquí no es casual: primero se comprueba que las variables *existen* y
son plausibles, y sólo después se mira su forma. Un histograma de una columna
con un 40% de huecos y un centinela `-999` describe el centinela, no la variable.

### 1.1 Descriptivos y faltantes

In [ ]:
# -> src/packagename/data/quality.py cuando el conjunto de comprobaciones se
# estabilice: es exactamente el tipo de resumen que se quiere poder recalcular
# desde la línea de comandos sobre cualquier tabla.
def describe_quality(
    frame: pd.DataFrame,
    ranges: dict[str, tuple[float, float]],
    near_constant: float,
) -> pd.DataFrame:
    rows = []
    for name, column in frame.items():
        values = column.dropna()
        # `value_counts(normalize=True)` sobre una columna continua da fracciones
        # minúsculas; el máximo sólo se acerca a 1 si la variable repite valor,
        # que es justo lo que se busca detectar.
        top_share = values.value_counts(normalize=True).max() if len(values) else np.nan
        low, high = ranges.get(name, (-np.inf, np.inf))
        outside = int(((values < low) | (values > high)).sum())
        rows.append(
            {
                "variable": name,
                "n": len(values),
                "missing_pct": 100 * column.isna().mean(),
                "mean": values.mean(),
                "std": values.std(),
                "min": values.min(),
                "p50": values.median(),
                "max": values.max(),
                "skew": values.skew(),
                "kurtosis": values.kurtosis(),
                "top_value_share": top_share,
                "near_constant": bool(top_share >= near_constant),
                "outside_physical_range": outside,
                "has_range_declared": name in ranges,
            }
        )
    return pd.DataFrame(rows).set_index("variable")


quality = describe_quality(numeric, PHYSICAL_RANGES, NEAR_CONSTANT)
quality.round(3)

In [ ]:
# Variables que ya se pueden descartar sin discutir: sin variabilidad, o con
# tantos huecos que cualquier imputación sería el modelo y no el dato.
print("Casi constantes:", quality.index[quality["near_constant"]].tolist())
print("Faltantes > 20%:", quality.index[quality["missing_pct"] > 20].tolist())
print("Fuera de rango:", quality.index[quality["outside_physical_range"] > 0].tolist())
print("Sin rango declarado:", quality.index[~quality["has_range_declared"]].tolist())

### 1.2 Faltantes en el tiempo

El porcentaje global de huecos esconde lo que de verdad decide la estrategia de
imputación: si los huecos están repartidos o concentrados. Un 5% disperso se
interpola; un 5% que es un mes entero caído no se interpola, se excluye.

In [ ]:
monthly_missing = numeric.isna().resample("MS").mean().mul(100)

fig, ax = plt.subplots(figsize=(11, 0.35 * len(numeric.columns) + 2))
mesh = ax.pcolormesh(
    monthly_missing.index,
    np.arange(len(monthly_missing.columns)),
    monthly_missing.to_numpy().T,
    cmap="magma_r",
    vmin=0,
    vmax=100,
)
ax.set_yticks(np.arange(len(monthly_missing.columns)), monthly_missing.columns)
ax.set_title("Porcentaje de datos faltantes por variable y mes")
fig.colorbar(mesh, ax=ax, label="% faltante")
remove_grid(ax)
savefig(fig, f"{FIG}/missing_by_month.png")

### 1.3 Continuidad y resolución del índice

La resolución nominal ("horaria", "trihoraria") es un dato del catálogo; la real
es lo que dicen las diferencias entre instantes consecutivos. Cualquier ACF,
espectro o descomposición STL asume una malla regular, así que esta celda va
antes que todas ellas.

In [ ]:
steps = data.index.to_series().diff().dropna()
step = steps.mode().iloc[0]
gaps = steps[steps > step]

print(f"Paso dominante: {step}  ({100 * (steps == step).mean():.2f}% de los intervalos)")
print(f"Instantes duplicados: {int(data.index.duplicated().sum())}")
print(f"Discontinuidades: {len(gaps)}")
if len(gaps):
    print(gaps.sort_values(ascending=False).head(10))
    print(f"Cobertura real: {100 * len(data) * step / (data.index.max() - data.index.min()):.2f}%")

### 1.4 Distribuciones

Histograma y KDE sobre los mismos ejes: el histograma no oculta la
discretización de la variable y el KDE no oculta la forma de las colas. Las
líneas verticales marcan el rango físico declarado, para que un valor imposible
se vea en lugar de tener que buscarse en una tabla.

In [ ]:
columns = [TARGET, *features] if TARGET in numeric.columns else features
ncols = 3
nrows = int(np.ceil(len(columns) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 2.8 * nrows))
for ax, name in zip(np.ravel(axes), columns, strict=False):
    values = numeric[name].dropna().to_numpy()
    ax.hist(values, bins=60, density=True, color=NEUTRALS["light"], edgecolor="none")
    # gaussian_kde falla con varianza nula, que es exactamente el caso de las
    # variables casi constantes detectadas arriba.
    if values.std() > 0:
        grid = np.linspace(values.min(), values.max(), 400)
        ax.plot(grid, stats.gaussian_kde(values)(grid), color=COLOR_NAMES["cornflower blue"])
    for bound in PHYSICAL_RANGES.get(name, ()):
        if values.min() <= bound <= values.max():
            ax.axvline(bound, color=COLOR_NAMES["crimson"], ls="--", lw=0.8)
    ax.set_title(name)
    ax.set_yticks([])

for ax in np.ravel(axes)[len(columns) :]:
    ax.set_visible(False)

fig.suptitle("Distribuciones marginales")
savefig(fig, f"{FIG}/distributions.png")

### 1.5 Outliers

Dos criterios, porque miden cosas distintas y discrepar es informativo: el rango
intercuartílico (robusto, no supone forma) y el z-score modificado sobre la MAD
(robusto, sí supone unimodalidad). En oleaje conviene mirar los "outliers" uno a
uno antes de tocarlos: un percentil 99.9 de altura de ola es un temporal, es
decir, precisamente el régimen que el modelo tiene que acertar.

In [ ]:
# -> src/packagename/data/quality.py junto con describe_quality.
def flag_outliers(frame: pd.DataFrame, iqr_factor: float = 3.0, z_cut: float = 3.5) -> pd.DataFrame:
    rows = []
    for name, column in frame.items():
        values = column.dropna()
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        by_iqr = ((values < q1 - iqr_factor * iqr) | (values > q3 + iqr_factor * iqr)).mean()
        mad = stats.median_abs_deviation(values, scale="normal")
        # scale="normal" ya reescala la MAD a la sigma de una normal, así que el
        # corte se compara directamente con un z-score.
        by_mad = (np.abs(values - values.median()) / mad > z_cut).mean() if mad > 0 else 0.0
        rows.append(
            {
                "variable": name,
                "iqr_pct": 100 * by_iqr,
                "modified_z_pct": 100 * by_mad,
                "p99_9": values.quantile(0.999),
                "max": values.max(),
            }
        )
    return pd.DataFrame(rows).set_index("variable")


outliers = flag_outliers(numeric)
outliers.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
standardised = (numeric - numeric.median()) / numeric.std()
ax.boxplot(
    [standardised[name].dropna() for name in standardised.columns],
    tick_labels=standardised.columns,
    flierprops={"marker": ".", "markersize": 2, "alpha": 0.3},
)
ax.set_ylabel("desviaciones típicas respecto a la mediana")
ax.set_title("Dispersión comparada (variables estandarizadas)")
ax.tick_params(axis="x", rotation=90)
savefig(fig, f"{FIG}/outliers_boxplot.png")

## 2. Análisis temporal

Todo lo que sigue asume la malla regular comprobada en 1.3. `asfreq` la impone
explícitamente: convierte las discontinuidades en `NaN`, que es un hueco que las
funciones de abajo pueden reconocer, en lugar de un salto silencioso que
falsearía cada retardo.

In [ ]:
series = data[TARGET] if GROUP is None else data.loc[data[GROUP] == data[GROUP].iloc[0], TARGET]
series = series.asfreq(step)
print(f"{len(series):,} instantes en malla regular; {series.isna().sum():,} huecos")

### 2.1 Evolución de la media y tendencia

La regresión es de Theil-Sen y no de mínimos cuadrados: en series de oleaje unos
pocos temporales bastan para inclinar una recta OLS, y lo que interesa saber es
si hay deriva en el cuerpo de la distribución. El intervalo de confianza que
devuelve es lo que decide si la pendiente merece llamarse tendencia.

In [ ]:
monthly = series.resample("MS").mean()
annual = series.resample("YS").mean()

elapsed_years = ((monthly.index - monthly.index[0]).days / 365.25).to_numpy()
observed = monthly.notna().to_numpy()
slope, intercept, low, high = stats.theilslopes(monthly[observed], elapsed_years[observed])

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(monthly.index, monthly, lw=0.7, color=NEUTRALS["light"], label="media mensual")
ax.plot(annual.index, annual, marker="o", color=COLOR_NAMES["cornflower blue"], label="media anual")
ax.plot(
    monthly.index,
    intercept + slope * elapsed_years,
    color=COLOR_NAMES["crimson"],
    label=f"Theil-Sen: {slope:+.4f}/año [{low:+.4f}, {high:+.4f}]",
)
ax.set_ylabel(TARGET)
ax.set_title(f"Evolución temporal de {TARGET}")
ax.legend()
savefig(fig, f"{FIG}/trend.png")

significant = "sí" if low * high > 0 else "no"
print(f"¿El IC excluye la pendiente nula? {significant}")

### 2.2 Ciclos y estacionalidad

El ciclo anual y el diario se miran por separado porque tienen causas distintas
y consecuencias distintas sobre el modelo: el anual justifica incluir el mes o
un par de armónicos como variable; el diario, en oleaje, suele ser débil, y su
ausencia es un resultado que conviene comprobar en lugar de suponer.

In [ ]:
SEASONS = {
    12: "DJF",
    1: "DJF",
    2: "DJF",
    3: "MAM",
    4: "MAM",
    5: "MAM",
    6: "JJA",
    7: "JJA",
    8: "JJA",
    9: "SON",
    10: "SON",
    11: "SON",
}
ORDER = ["DJF", "MAM", "JJA", "SON"]

frame = series.to_frame(TARGET)
frame["month"] = frame.index.month
frame["hour"] = frame.index.hour
frame["season"] = frame["month"].map(SEASONS)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

by_month = frame.groupby("month")[TARGET]
axes[0].errorbar(by_month.mean().index, by_month.mean(), yerr=by_month.std(), marker="o", capsize=3)
axes[0].set_xlabel("mes")
axes[0].set_title("Ciclo anual")

by_hour = frame.groupby("hour")[TARGET]
axes[1].errorbar(by_hour.mean().index, by_hour.mean(), yerr=by_hour.std(), marker="o", capsize=3)
axes[1].set_xlabel("hora UTC")
axes[1].set_title("Ciclo diario")

axes[2].boxplot(
    [frame.loc[frame["season"] == s, TARGET].dropna() for s in ORDER], tick_labels=ORDER
)
axes[2].set_title("Por estación")

for ax in axes:
    ax.set_ylabel(TARGET)
fig.suptitle(f"Ciclos de {TARGET}")
savefig(fig, f"{FIG}/cycles.png")

frame.groupby("season")[TARGET].describe().reindex(ORDER).round(3)

### 2.3 Autocorrelación

La ACF dice cuánta memoria tiene la serie, y de ahí sale una consecuencia
concreta para la validación: dos instantes separados por menos de la longitud de
decorrelación no son observaciones independientes, así que una validación cruzada
que los reparta entre *train* y *test* mide memoria y no capacidad predictiva.
Ese número se usa en `04` para dimensionar el hueco entre particiones.

In [ ]:
# acf/pacf no aceptan NaN: se interpolan sólo para este cálculo, sin tocar
# `series`, y los huecos largos se dejan fuera del ajuste.
filled = series.interpolate(limit=3).dropna()
nlags = min(3 * MAX_LAG, len(filled) // 2 - 1)
acf_values, acf_ci = acf(filled, nlags=nlags, alpha=0.05, fft=True)
pacf_values, pacf_ci = pacf(filled, nlags=min(nlags, 60), alpha=0.05)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for ax, values, confidence, label in (
    (axes[0], acf_values, acf_ci, "ACF"),
    (axes[1], pacf_values, pacf_ci, "PACF"),
):
    lags = np.arange(len(values))
    ax.vlines(lags, 0, values, color=COLOR_NAMES["cornflower blue"])
    ax.fill_between(lags, confidence[:, 0] - values, confidence[:, 1] - values, alpha=0.2)
    ax.axhline(0, color=NEUTRALS["dark_slate"], lw=0.8)
    ax.set_xlabel("lag (pasos)")
    ax.set_title(f"{label} de {TARGET}")
savefig(fig, f"{FIG}/acf_pacf.png")

crossing = np.argmax(acf_values < 1 / np.e)
print(f"La ACF cae por debajo de 1/e en el lag {crossing} ({crossing * step})")

### 2.4 Espectro

Welch en lugar de una FFT desnuda: promedia periodogramas sobre ventanas
solapadas, así que los picos que sobreviven son los que se repiten a lo largo de
la serie y no los que un tramo concreto impuso. El eje se etiqueta en período y
no en frecuencia porque lo que se busca son los ciclos ya conocidos --
diario, semianual, anual -- y lo interesante es lo que aparece *además* de ellos.

In [ ]:
steps_per_day = pd.Timedelta("1D") / step
frequencies, power = signal.welch(
    filled - filled.mean(),
    fs=steps_per_day,  # ciclos por día
    nperseg=min(len(filled), int(steps_per_day * 365 * 2)),
)

fig, ax = plt.subplots(figsize=(9, 4))
positive = frequencies > 0
ax.loglog(1 / frequencies[positive], power[positive], lw=0.8)
for period, label in ((1, "diario"), (182.6, "semianual"), (365.25, "anual")):
    ax.axvline(period, color=COLOR_NAMES["crimson"], ls="--", lw=0.8)
    ax.text(period, ax.get_ylim()[1], label, rotation=90, va="top", fontsize=8)
ax.set_xlabel("período (días)")
ax.set_ylabel("densidad espectral")
ax.set_title(f"Espectro de Welch de {TARGET}")
savefig(fig, f"{FIG}/spectrum.png")

### 2.5 Descomposición STL

STL separa tendencia, estacionalidad y residuo sin suponer que la estacionalidad
sea constante en amplitud, que es lo que la hace apropiada aquí. La varianza del
residuo respecto al total es una cota informal de lo que un modelo puede
aspirar a explicar con el calendario solo: si la estacionalidad ya se lleva la
mayor parte, un predictor que no la supere no está aportando nada.

In [ ]:
daily = series.resample("D").mean().interpolate(limit=7).dropna()
stl = STL(daily, period=365, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
for ax, component, label in (
    (axes[0], stl.observed, "observado"),
    (axes[1], stl.trend, "tendencia"),
    (axes[2], stl.seasonal, "estacionalidad"),
    (axes[3], stl.resid, "residuo"),
):
    ax.plot(component.index, component, lw=0.6)
    ax.set_ylabel(label)
fig.suptitle(f"Descomposición STL de {TARGET} (media diaria)")
savefig(fig, f"{FIG}/stl.png")

total = stl.observed.var()
print(f"Varianza explicada por la estacionalidad: {100 * stl.seasonal.var() / total:.1f}%")
print(f"Varianza explicada por la tendencia:      {100 * stl.trend.var() / total:.1f}%")
print(f"Varianza en el residuo:                   {100 * stl.resid.var() / total:.1f}%")

## 3. Relación entre variables

### 3.1 Correlaciones

Pearson y Spearman a la vez, porque su diferencia es el dato útil: cuando
Spearman es claramente mayor, la relación es monótona pero no lineal, y eso
dice que un baseline lineal va a quedarse corto antes de haberlo entrenado.

In [ ]:
pearson = numeric.corr(method="pearson")
spearman = numeric.corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, matrix, label in ((axes[0], pearson, "Pearson"), (axes[1], spearman, "Spearman")):
    mesh = ax.pcolormesh(matrix.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(np.arange(len(matrix)) + 0.5, matrix.columns, rotation=90)
    ax.set_yticks(np.arange(len(matrix)) + 0.5, matrix.index)
    ax.set_title(label)
    ax.invert_yaxis()
    remove_grid(ax)
fig.colorbar(mesh, ax=axes, label="correlación")
savefig(fig, f"{FIG}/correlation_matrices.png")

In [ ]:
with_target = pd.DataFrame(
    {
        "pearson": pearson[TARGET].drop(TARGET),
        "spearman": spearman[TARGET].drop(TARGET),
    }
)
with_target["no_linealidad"] = with_target["spearman"].abs() - with_target["pearson"].abs()
with_target.reindex(with_target["spearman"].abs().sort_values(ascending=False).index).round(3)

### 3.2 Multicolinealidad (VIF)

En datos meteorológicos la colinealidad es la regla, no la excepción: la presión
a nivel del mar, el viento y el oleaje son manifestaciones del mismo sistema. El
VIF no dice que una variable sea inútil, dice que su coeficiente no es
interpretable por separado -- que es la razón por la que en `03` la selección no
se apoya sólo en modelos lineales.

In [ ]:
# -> src/packagename/features/diagnostics.py
def variance_inflation(frame: pd.DataFrame) -> pd.Series:
    # VIF_i = 1 / (1 - R2_i) al regresar la variable i sobre las demás, y ese R2
    # sale de la diagonal de la inversa de la matriz de correlación, sin ajustar
    # ni una regresión.
    clean = frame.dropna()
    correlation = clean.corr().to_numpy()
    inverse = np.linalg.pinv(correlation)
    return pd.Series(np.diag(inverse), index=clean.columns).sort_values(ascending=False)


vif = variance_inflation(numeric[features])
vif.round(2).to_frame("VIF")

### 3.3 Información mutua

La correlación no ve una relación en forma de U; la información mutua sí. El
contraste entre este ranking y el de Spearman es lo que señala qué variables
merecen un modelo no lineal. Se estandariza dividiendo por el máximo: la MI está
en nats y su valor absoluto no es comparable entre datasets.

In [ ]:
complete = numeric[[TARGET, *features]].dropna()
mutual = pd.Series(
    mutual_info_regression(
        complete[features],
        complete[TARGET],
        random_state=settings.random_seed,
    ),
    index=features,
).sort_values(ascending=False)

comparison = pd.DataFrame(
    {
        "mi": mutual,
        "mi_relativa": mutual / mutual.max(),
        "spearman_abs": spearman[TARGET].drop(TARGET).abs(),
    }
).sort_values("mi", ascending=False)

fig, ax = plt.subplots(figsize=(9, 0.3 * len(features) + 2))
positions = np.arange(len(comparison))
ax.barh(positions - 0.2, comparison["mi_relativa"], height=0.4, label="MI (relativa)")
ax.barh(positions + 0.2, comparison["spearman_abs"], height=0.4, label="|Spearman|")
ax.set_yticks(positions, comparison.index)
ax.invert_yaxis()
ax.legend()
ax.set_title(f"Dependencia con {TARGET}: lineal frente a no lineal")
savefig(fig, f"{FIG}/mutual_information.png")

comparison.round(3)

### 3.4 Estructura de grupos

Reordenar la matriz de correlación por un clustering jerárquico convierte una
cuadrícula ilegible en bloques: cada bloque es un grupo de variables que dicen
casi lo mismo. La distancia es `1 - |rho|`, así que dos variables
anticorreladas quedan juntas, que es lo correcto -- son igual de redundantes.

In [ ]:
distance = (1 - spearman.loc[features, features].abs()).to_numpy()
# squareform exige diagonal exactamente nula; 1 - |rho(x, x)| puede dejar un
# residuo de coma flotante que la haría fallar.
np.fill_diagonal(distance, 0.0)
linkage = hierarchy.linkage(squareform(distance, checks=False), method="average")
order = [features[index] for index in hierarchy.leaves_list(linkage)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), width_ratios=[1, 1.4])
hierarchy.dendrogram(linkage, labels=features, ax=axes[0], color_threshold=0.3, leaf_rotation=90)
axes[0].axhline(0.3, color=COLOR_NAMES["crimson"], ls="--", lw=0.8)
axes[0].set_ylabel("1 - |Spearman|")
axes[0].set_title("Dendrograma de variables")

clustered = spearman.loc[order, order]
mesh = axes[1].pcolormesh(clustered.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set_xticks(np.arange(len(order)) + 0.5, order, rotation=90)
axes[1].set_yticks(np.arange(len(order)) + 0.5, order)
axes[1].invert_yaxis()
axes[1].set_title("Correlación reordenada por grupo")
remove_grid(axes[1])
fig.colorbar(mesh, ax=axes[1])
savefig(fig, f"{FIG}/clustered_correlation.png")

groups = pd.Series(
    hierarchy.fcluster(linkage, t=0.3, criterion="distance"), index=features
).sort_values()
groups.to_frame("grupo")

### 3.5 Pares más relevantes

Una matriz de dispersión de *todas* las variables no se lee. Se limita a las que
la MI ha señalado, que es el orden en que uno querría mirarlas de todas formas.

In [ ]:
top = comparison.index[:5].tolist()
axes = pd.plotting.scatter_matrix(
    complete[[TARGET, *top]].sample(min(3000, len(complete)), random_state=settings.random_seed),
    figsize=(10, 10),
    diagonal="kde",
    alpha=0.2,
    s=4,
)
for ax in np.ravel(axes):
    ax.xaxis.label.set_rotation(45)
    ax.yaxis.label.set_rotation(0)
    ax.yaxis.labelpad = 30
savefig(np.ravel(axes)[0].get_figure(), f"{FIG}/scatter_matrix.png")

### 3.6 Correlación cruzada con retardos

La celda que más veces cambia el diseño de las variables. Una variable
atmosférica actúa sobre el oleaje con retraso -- el mar tarda en responder al
viento y el *swell* llega desde lejos -- así que el lag que maximiza la
correlación no es 0, y es el lag que `02` tendrá que construir. Sólo se miran
retardos positivos: usar el futuro para predecir el presente es fuga de
información, no una variable.

In [ ]:
regular = numeric.asfreq(step) if GROUP is None else numeric
lags = np.arange(0, MAX_LAG + 1)
cross = pd.DataFrame(
    {name: [regular[name].shift(lag).corr(regular[TARGET]) for lag in lags] for name in features},
    index=lags,
)
cross.index.name = "lag"

fig, ax = plt.subplots(figsize=(10, 4.5))
for name in comparison.index[:8]:
    ax.plot(cross.index, cross[name], marker=".", label=name)
ax.axhline(0, color=NEUTRALS["dark_slate"], lw=0.8)
ax.set_xlabel(f"retardo aplicado al predictor (1 paso = {step})")
ax.set_ylabel(f"correlación con {TARGET}(t)")
ax.set_title("Correlación cruzada con retardos")
ax.legend(ncols=2, fontsize=8)
savefig(fig, f"{FIG}/cross_correlation.png")

best_lags = pd.DataFrame(
    {
        "lag_optimo": cross.abs().idxmax(),
        "corr_en_lag_optimo": [
            cross.loc[cross[name].abs().idxmax(), name] for name in cross.columns
        ],
        "corr_en_lag_0": cross.loc[0],
    }
)
best_lags["ganancia"] = best_lags["corr_en_lag_optimo"].abs() - best_lags["corr_en_lag_0"].abs()
best_lags.sort_values("ganancia", ascending=False).round(3)

## 4. Análisis espacial

Esta sección sólo se ejecuta si `GRID_FILE` apunta a un campo gridded. Es la
parte que da sentido a "regionalización": los mapas de media y desviación típica
describen el dominio, y las EOF dicen cuántos modos independientes hacen falta
para representarlo -- que es la respuesta cuantitativa a en cuántas regiones
tiene sentido dividirlo.

Los mapas usan `pcolormesh` sobre latitud y longitud, sin proyección. Para
figuras de publicación conviene añadir `cartopy` y proyectar; para explorar, una
proyección mal elegida sólo estorba.

In [ ]:
if GRID_FILE is None:
    print("GRID_FILE = None: sección espacial omitida")
    grid = None
else:
    import xarray as xr

    grid = xr.open_dataset(settings.paths.raw / GRID_FILE)
    print(grid)

In [ ]:
if grid is not None:
    field = grid[TARGET]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, values, label, cmap in (
        (axes[0], field.mean("time"), "media", "viridis"),
        (axes[1], field.std("time"), "desviación típica", "magma"),
        (axes[2], field.std("time") / field.mean("time"), "coef. de variación", "cividis"),
    ):
        mesh = ax.pcolormesh(field["longitude"], field["latitude"], values, cmap=cmap)
        fig.colorbar(mesh, ax=ax)
        ax.set_title(f"{TARGET}: {label}")
        ax.set_xlabel("longitud")
        ax.set_ylabel("latitud")
        remove_grid(ax)
    savefig(fig, f"{FIG}/spatial_summary.png")

In [ ]:
if grid is not None:
    from sklearn.decomposition import PCA

    # Las EOF se calculan sobre anomalías: sin quitar la media, el primer modo es
    # siempre el campo medio y no explica variabilidad ninguna.
    anomalies = (field - field.mean("time")).stack(cell=("latitude", "longitude"))
    matrix = anomalies.to_numpy()
    usable = ~np.isnan(matrix).any(axis=0)

    pca = PCA(n_components=10, random_state=settings.random_seed)
    scores = pca.fit_transform(matrix[:, usable])
    explained = 100 * pca.explained_variance_ratio_

    for index, share in enumerate(explained, start=1):
        print(f"EOF{index} = {share:.1f}%   (acumulado {explained[:index].sum():.1f}%)")

In [ ]:
if grid is not None:
    shape = (field.sizes["latitude"], field.sizes["longitude"])
    fig, axes = plt.subplots(2, 4, figsize=(16, 7))
    for index, ax in enumerate(np.ravel(axes)[:4]):
        loading = np.full(usable.size, np.nan)
        loading[usable] = pca.components_[index]
        limit = np.nanmax(np.abs(loading))
        mesh = ax.pcolormesh(
            field["longitude"],
            field["latitude"],
            loading.reshape(shape),
            cmap="RdBu_r",
            vmin=-limit,
            vmax=limit,
        )
        fig.colorbar(mesh, ax=ax)
        ax.set_title(f"EOF{index + 1} ({explained[index]:.1f}%)")
        remove_grid(ax)

    for index, ax in enumerate(np.ravel(axes)[4:]):
        ax.plot(field["time"], scores[:, index], lw=0.5)
        ax.set_title(f"Serie temporal del modo {index + 1}")

    fig.suptitle("Modos espaciales y sus coeficientes")
    savefig(fig, f"{FIG}/eof_loadings.png")

## 5. Tabla resumen

Una fila por variable con todo lo anterior junto. Este fichero es el que se cita
en `03` al justificar por qué una variable entra o sale: un ranking que no se
puede rastrear hasta los números que lo produjeron no es un argumento.

`RF importance` no se calcula aquí a propósito. Requiere ajustar un modelo, y un
modelo ajustado sobre datos sin partir ni escalar da una importancia que no se
puede comparar con nada. Se añade en `03`, sobre la matriz que sale de `02`.

In [ ]:
summary = quality.join(
    [
        outliers[["iqr_pct", "modified_z_pct"]],
        with_target.rename(columns={"pearson": "corr_pearson", "spearman": "corr_spearman"}),
        vif.to_frame("vif"),
        comparison[["mi"]],
        best_lags[["lag_optimo", "ganancia"]],
        groups.to_frame("grupo_correlacion"),
    ],
    how="left",
)
summary = summary.sort_values("mi", ascending=False)
write_table(summary.reset_index(), TABLES / "variable_summary.csv")
summary.round(3)

## Conclusiones

Rellenar a mano, en prosa, antes de pasar a `02`. Sin esto el notebook es una
galería de figuras que nadie relee. Las preguntas que hay que dejar contestadas:

1. **Qué variables se descartan ya y por qué** -- casi constantes, demasiados
   huecos, o fuera de rango físico.
2. **Cómo están distribuidos los huecos** y qué estrategia de imputación admiten
   (§1.2): dispersos se interpolan, concentrados se excluyen.
3. **Qué retardos hay que construir** (§3.6), variable por variable.
4. **Qué grupos de variables son redundantes** (§3.4) y cuál representa a cada
   grupo.
5. **Cuánta memoria tiene la serie** (§2.3), porque de ahí sale el hueco entre
   particiones en `04`.
6. **Cuántos modos espaciales hacen falta** (§4) si el problema es regional.